# Wine Reviews — Exploratory Data Analysis

Purely exploratory notebook, not part of the final pipeline.

**Goals**
- Map missing values across columns
- Inspect distributions of `retail` (target) and `rating`
- Surface top categorical values (country, wine_type, varietal, ...)
- Compute numeric correlations
- Visualize price vs rating

Loaded from the raw Bronze parquet (CSV-as-parquet, all strings) — light
numeric coercion is applied here. The authoritative cleaning lives in
`1_ingest_pd.ipynb` and will move to `etl/3_clean.py` (Silver) next.

In [18]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

BRONZE_PATH = r"..\..\.data\wine_reviews_bronze.parquet"

df = pd.read_parquet(BRONZE_PATH)
print(f"Shape: {df.shape}")
df.dtypes

Shape: (135211, 22)


name               str
brand              str
company            str
vintage            str
drink_type         str
wine_type          str
varietal_label     str
alcohol            str
bottle_size        str
case_production    str
country            str
state              str
appellation        str
designation        str
retail             str
rating             str
reviewer           str
review             str
date_of_review     str
date_received      str
pub_date_web       str
slug               str
dtype: object

## Light numeric coercion

Bronze is all strings — coerce the six numeric columns so distributions
and correlations work. Anything non-numeric becomes NaN.

In [19]:
numeric_cols = ["alcohol", "vintage", "case_production", "retail", "rating", "bottle_size"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].dtypes

alcohol            float64
vintage            float64
case_production      int64
retail             float64
rating               int64
bottle_size        float64
dtype: object

## 1. Missing values

Per-column null counts and percentages.

In [20]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = (
    pd.DataFrame({"nulls": missing, "pct": missing_pct})
    .query("nulls > 0")
    .sort_values("nulls", ascending=False)
)
missing_df

,nulls,pct
bottle_size,129468,95.75
designation,32421,23.98
vintage,4619,3.42
reviewer,244,0.18
review,17,0.01
varietal_label,12,0.01
company,2,0.00


### Missing-value heatmap

Each row = column, each column = sample of records. Red stripes show
where nulls cluster — useful for spotting structurally-missing groups
(e.g. `case_production` mostly missing for non-US wines).

In [21]:
sample = df.sample(min(5000, len(df)), random_state=42).sort_index()
fig = px.imshow(
    sample.isna().T.astype(int),
    aspect="auto",
    color_continuous_scale=["white", "tomato"],
    title="Missing-value heatmap (5k sample; red = missing)",
    labels={"x": "row sample", "y": "column", "color": "missing"},
)
fig.update_layout(height=700, coloraxis_showscale=False)
fig.show()

## 2. Retail price distribution

`retail` is the target variable. Expect strong right-skew → confirm a
`log` transform is appropriate for modelling.

In [22]:
retail = df["retail"].dropna()
print(retail.describe().round(2))
print()
print(f"Skewness (raw):    {retail.skew():.2f}")
print(f"Skewness (log1p):  {np.log1p(retail).skew():.2f}") # type: ignore
print(f"Zeros:             {(retail == 0).sum():,}")
print(f"Above $500:        {(retail > 500).sum():,}")

count    135211.00
mean         41.85
std          71.75
min           0.00
25%          19.00
50%          30.00
75%          50.00
max        9999.99
Name: retail, dtype: float64

Skewness (raw):    60.15
Skewness (log1p):  -1.57
Zeros:             8,035
Above $500:        168


In [23]:
low, high = retail.quantile([0.01, 0.99])
fig = px.histogram(
    retail[(retail >= low) & (retail <= high)],
    nbins=100,
    title=f"Retail price — 1st-99th pct (${low:.0f}-${high:.0f})",
    labels={"value": "Retail (USD)"},
    marginal="box",
    opacity=0.85,
)
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

In [24]:
log_retail = np.log10(retail[retail > 0])
fig = px.histogram(
    log_retail,
    nbins=80,
    title="log10(retail) — much closer to normal, confirms log target",
    labels={"value": "log10(retail USD)"},
    marginal="box",
    opacity=0.85,
)
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

## 3. Rating distribution

Wine Enthusiast publishes mostly 80+ ratings — expect a narrow range
centred around ~90 with a hard floor at 80.

In [25]:
rating = df["rating"].dropna()
print(rating.describe().round(2))
print(f"Below 80: {(rating < 80).sum():,}  (likely cast errors)")

count    135211.00
mean         90.34
std           2.75
min          22.00
25%          88.00
50%          90.00
75%          92.00
max         100.00
Name: rating, dtype: float64
Below 80: 10  (likely cast errors)


In [26]:
fig = px.histogram(
    rating[rating >= 80],
    nbins=21,
    title="Rating distribution (80-100)",
    labels={"value": "Rating (pts)"},
    marginal="box",
    opacity=0.85,
)
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

## 4. Top categorical values

Cardinality and dominance check — which countries / types / varietals
dominate the dataset, and how long the long tail is.

In [27]:
cat_cols = ["country", "wine_type", "varietal_label", "state", "reviewer", "appellation"]

print(f"{'column':<18} {'unique':>8} {'top':>30} {'top_count':>10} {'top_pct':>8}")
print("-" * 80)
for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    top = vc.index[0]
    top_count = vc.iloc[0]
    top_pct = top_count / len(df) * 100
    print(f"{col:<18} {df[col].nunique():>8,} {str(top)[:30]:>30} {top_count:>10,} {top_pct:>7.1f}%")

column               unique                            top  top_count  top_pct
--------------------------------------------------------------------------------
country                  43                            USA     50,073    37.0%
wine_type                10                            Red     80,220    59.3%
varietal_label        1,047                     Pinot Noir     15,176    11.2%
state                    45                          False     79,521    58.8%
reviewer                 50                           R.V.     24,378    18.0%
appellation           1,525                     California      2,929     2.2%


In [28]:
TOP_N = 15
for col in ["country", "wine_type", "varietal_label", "reviewer"]:
    counts = df[col].value_counts().head(TOP_N)
    fig = px.bar(
        x=counts.values,
        y=counts.index,
        orientation="h",
        title=f"Top {TOP_N} {col} by review count",
        labels={"x": "count", "y": col},
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(height=420)
    fig.show()

## 5. Numeric correlations

Pearson correlations between numeric features. Look for the
`rating`-`retail` signal in particular.

In [29]:
corr = df[numeric_cols].corr(method="pearson").round(3)
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Numeric feature correlations (Pearson)",
)
fig.update_layout(height=520, width=620)
fig.show()
corr

,alcohol,vintage,case_production,retail,rating,bottle_size
alcohol,1.000,-0.004,-0.000,0.001,0.001,0.063
vintage,-0.004,1.000,0.007,-0.143,-0.083,0.068
case_production,-0.000,0.007,1.000,-0.006,-0.012,0.001
retail,0.001,-0.143,-0.006,1.000,0.292,-0.044
rating,0.001,-0.083,-0.012,0.292,1.000,-0.024
bottle_size,0.063,0.068,0.001,-0.044,-0.024,1.000


In [30]:
# log-retail correlations — usually stronger than raw retail
df_log = df[numeric_cols].copy()
df_log["log_retail"] = np.log1p(df_log["retail"])
log_corr = df_log.corr(method="pearson")["log_retail"].drop("log_retail").round(3)
log_corr.sort_values(ascending=False)

retail             0.447
rating             0.357
alcohol            0.003
case_production   -0.008
bottle_size       -0.021
vintage           -0.140
Name: log_retail, dtype: float64

## 6. Price vs Rating

Sample-based scatter (full dataset overplots heavily). Y-axis log to
spread the long tail; colour by `wine_type`.

In [31]:
mask = df["retail"].between(*df["retail"].quantile([0.01, 0.99])) & df["rating"].notna()
sample = df[mask].sample(min(15000, mask.sum()), random_state=42)

fig = px.scatter(
    sample,
    x="rating",
    y="retail",
    color="wine_type",
    opacity=0.45,
    title="Retail vs Rating (15k sample, 1-99 pct of retail)",
    labels={"retail": "Retail (USD, log)", "rating": "Rating"},
)
fig.update_yaxes(type="log")
fig.update_layout(height=560)
fig.show()

In [32]:
# Mean and median retail per rating bucket
by_rating = (
    df.dropna(subset=["retail", "rating"])
    .assign(rating=lambda d: d["rating"].astype(int))
    .groupby("rating")["retail"]
    .agg(["count", "mean", "median"])
    .round(2)
)
by_rating

,count,mean,median
rating,,,
22,5,18.40,18.0
44,5,16.40,15.0
80,54,19.17,15.5
81,56,16.71,14.5
82,253,20.64,17.0
83,522,18.15,15.0
84,1264,18.33,15.0
85,2679,18.60,15.0
86,5117,20.63,16.0


In [33]:
fig = px.line(
    by_rating.reset_index(),
    x="rating",
    y=["mean", "median"],
    title="Retail by rating — mean vs median",
    labels={"value": "Retail (USD)", "variable": "stat"},
    markers=True,
)
fig.show()

## 7. ydata-profiling report (optional)

Runs the full profile and saves as HTML. Heavy — a couple of minutes
on 135k rows. Skip if you've already generated it.

In [34]:
try:
    from ydata_profiling import ProfileReport

    profile = ProfileReport(df, title="Wine Reviews EDA", explorative=True)
    profile.to_file(r"..\..\.data\wine_reviews_profile.html")
    print("Profile saved to .data/wine_reviews_profile.html")
except ImportError as exc:
    print(f"ydata-profiling not available: {exc}")

ydata-profiling not available: No module named 'pydantic_core._pydantic_core'


## Key findings

- **Retail is strongly right-skewed** (skew ~10+ raw, ~0 log) — train on
  `log_retail`. A long tail of $500+ bottles will distort RMSE if kept.
- **Rating is narrow (80-100) and centred at ~90** — publication bias.
  Variance is small but its correlation with `retail` is the dominant
  numeric signal (matches the XGBoost feature-importance from Sprint 1).
- **`case_production` is 29% missing** and has an extreme max (1e8) —
  cap aggressively in Silver, or treat missing as a category.
- **`designation` is 24% missing** — high-cardinality (47k unique). Best
  handled with target encoding or "none" bucket, not one-hot.
- **USA dominates** (~37% of reviews); top 5 countries cover >80%.
- **`reviewer` has only ~50 distinct values** with heavy concentration —
  small enough for ordinal/one-hot encoding, may carry stylistic bias.
- **Numeric correlations are weak overall** except `rating`-`retail`
  (~0.4 raw, stronger on log) — most of the signal will come from
  categoricals and the review text (Sprint 5/6).